Exercises 3 and 4: ΔlogLik per observation and condition coefficients (from the R outputs in results/)

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                       # draw to files only (avoids the notebook crash)
import matplotlib.pyplot as plt
from IPython.display import Image

RESULTS = Path("../results")
FIGURES = Path("../figures")
os.makedirs(FIGURES, exist_ok=True)

LAM = {"lam1": 1, "lam2": 2, "lam4": 4, "lam8": 8}

In [ ]:
def delta_loglik(name):
    """Read <name>_loglik.csv (written by the R script) and add ΔlogLik per observation."""
    df = pd.read_csv(RESULTS / f"{name}_loglik.csv")
    base = df.loc[df.model == "base", "loglik"].iloc[0]
    df["dll"] = (df.loglik - base) / df.n
    return df


def summary(df, title):
    """Best lambda, and difference between full and the best lossy model (log-likelihood units)."""
    lossy = df[df.model.isin(LAM)].copy()
    lossy["lam"] = lossy.model.map(LAM)
    lossy = lossy.sort_values("lam")
    full = df.loc[df.model == "full", "dll"].iloc[0]
    best = lossy.loc[lossy.dll.idxmax()]
    n = df.n.iloc[0]
    print(f"{title}: best lambda = {best.lam}, "
          f"full - best lossy = {(full - best.dll) * n:+.1f} log-likelihood units "
          f"({'full' if full > best.dll else 'lossy'} fits better)")
    return lossy, full


def plot_dll(ax, lossy, full, title):
    ax.plot(lossy.lam, lossy.dll, "o-", label="lossy")
    ax.axhline(full, ls="--", color="gray", label="full (λ = ∞)")
    ax.set_xscale("log", base=2)
    ax.set_xticks([1, 2, 4, 8])
    ax.set_xticklabels(["1", "2", "4", "8"])
    ax.set_xlabel("λ")
    ax.set_title(title, fontsize=10)

3(b), 3(c) Natural Stories: best λ and difference with full surprisal

In [ ]:
ns = delta_loglik("ns")
print(ns[["model", "loglik", "dll"]])
ns_lossy, ns_full = summary(ns, "Natural Stories (lambda_nat)")

fig, ax = plt.subplots(figsize=(5, 3.5))
plot_dll(ax, ns_lossy, ns_full, "Natural Stories")
ax.set_ylabel("ΔlogLik per observation")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "ns_dll_plot.pdf", bbox_inches="tight")
plt.savefig(FIGURES / "ns_dll_plot.png", dpi=150, bbox_inches="tight")
plt.close(fig)
Image(FIGURES / "ns_dll_plot.png")

4(b), 4(e) Both datasets side by side, plus Natural Stories distance ≥ 4 (y axes not shared)

In [ ]:
panels = [("ns", "Natural Stories"),
          ("ns_long", "Natural Stories, distance ≥ 4"),
          ("sc", "StructuralCost")]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, (name, title) in zip(axes, panels):
    lossy, full = summary(delta_loglik(name), title)   # prints lambda_sc for StructuralCost
    plot_dll(ax, lossy, full, title)
axes[0].set_ylabel("ΔlogLik per observation")
axes[0].legend()
plt.tight_layout()
plt.savefig(FIGURES / "dll_three_panels.pdf", bbox_inches="tight")
plt.savefig(FIGURES / "dll_three_panels.png", dpi=150, bbox_inches="tight")
plt.close(fig)
Image(FIGURES / "dll_three_panels.png")

4(d) Condition coefficients without and with lossy surprisal (λ = 8), saved by the R script in sc_cond_coefs.csv

In [ ]:
coefs = pd.read_csv(RESULTS / "sc_cond_coefs.csv")
coefs = coefs[coefs.term.str.startswith("condition")].copy()
coefs["condition"] = coefs.term.str.replace("condition", "", regex=False)
coefs["change"] = coefs.after - coefs.before
print(coefs[["condition", "before", "after", "change"]].round(4).to_string(index=False))

x = np.arange(len(coefs))
w = 0.38
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.bar(x - w / 2, coefs.before, w, label="without surprisal")
ax.bar(x + w / 2, coefs.after, w, label="with lossy surprisal (λ = 8)")
ax.set_xticks(x)
ax.set_xticklabels(coefs.condition)
ax.set_ylabel("coefficient vs Baseline (log RT)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "cond_coefs.pdf", bbox_inches="tight")
plt.savefig(FIGURES / "cond_coefs.png", dpi=150, bbox_inches="tight")
plt.close(fig)
Image(FIGURES / "cond_coefs.png")